In [ ]:
import pandas as pd
# import os
# os.chdir(r'D:\workspace\ai_22_work_bj\pandasProject\day04')

# 1. 演示分组对象

In [31]:
# 1. 准备动作, 读取数据, 获取df对象.
df = pd.read_csv('../data/uniqlo.csv')
df

,store_id,city,channel,gender_group,age_group,wkd_ind,product,customer,revenue,order,quant,unit_cost
0,658,深圳,线下,Female,25-29,Weekday,当季新品,4,796.0,4,4,59
1,146,杭州,线下,Female,25-29,Weekday,运动,1,149.0,1,1,49
2,70,深圳,线下,Male,>=60,Weekday,T恤,2,178.0,2,2,49
3,658,深圳,线下,Female,25-29,Weekday,T恤,1,59.0,1,1,49
4,229,深圳,线下,Male,20-24,Weekend,袜子,2,65.0,2,3,9
...,...,...,...,...,...,...,...,...,...,...,...,...
22288,146,杭州,线下,Female,30-34,Weekday,短裤,1,80.0,1,2,19
22289,430,成都,线下,Female,25-29,Weekend,T恤,1,79.0,1,1,49
22290,449,武汉,线下,Female,35-39,Weekday,T恤,1,158.0,1,2,49
22291,758,杭州,线下,Female,20-24,Weekday,袜子,1,26.0,1,1,9


In [ ]:
# 2. 基于一列进行分组, 获得: DataFrameGroupBy分组对象.
# 需求: 基于 顾客的性别 分组.
df.groupby('gender_group')              # <pandas.core.groupby.generic.DataFrameGroupBy object at 0x0000025029082030>
df.groupby('gender_group')['city']      # <pandas.core.groupby.generic.SeriesGroupBy object at 0x000002502AC56150>

# 用变量来接收 分组对象, 获取某个分组的信息.
df_gb = df.groupby('gender_group')      # 根据性别分组.
print(df_gb.get_group('Female'))        # 根据 分组名 获取到该组的信息

print(df_gb['city'])

In [ ]:
# 3. 基于多列进行分组, 获得: DataFrameGroupBy分组对象.
# 需求: 基于 顾客的性别 和 城市 分组.
df.groupby(['gender_group', 'city'])            # <pandas.core.groupby.generic.DataFrameGroupBy object at 0x000002502900FC50>
df.groupby(['gender_group', 'city'])['revenue'] # <pandas.core.groupby.generic.SeriesGroupBy object at 0x000002502AC563F0>

In [ ]:
# 4. 分组后, 获取到每个分组的第一条 或者 最后一条数据.
# 组后第一条: first()
# 组后最后一条: last()

# 4.1 变量记录, 分组对象.
df_gb = df.groupby(['gender_group', 'city'])
df_gb.first()
df_gb.last()

# 2. 演示分组聚合操作

In [ ]:
# 格式: df.groupby(['分组字段1', '字段2'...]).agg({'列名1':'聚合函数名', '列名2':'聚合函数名'...})
# 格式: df.pivot_table(index='行索引', columns='列', values='值', aggfunc='聚合函数名')

# 需求1: 按照城市分组, 计算每个城市的 客户数量. 
# 方式1: groupby() + 聚合函数.
df.groupby('city').customer.sum()
df.groupby('city').customer.agg('sum')      # 效果同上
df.groupby('city').agg({'customer':'sum'})  # 效果同上

# 方式2: pivot_table() 透视表
df.pivot_table(index='city', values='customer', aggfunc='sum')

In [ ]:
# 需求2: 按照城市, 性别分组, 计算每个城市的 客户数量. 
# 方式1: groupby() + 聚合函数.
df.groupby(['city', 'gender_group'])['customer'].sum()
# df.groupby(['city', 'gender_group']).customer.agg('sum')
df.groupby(['city', 'gender_group']).agg({'customer': 'sum'})

# # 方式2: pivot_table() 透视表
df.pivot_table(index=['city', 'gender_group'], values='customer', aggfunc='sum')

# 上述格式的变形写法, 类似于: 行列转置, 更直观的查看分组聚合结果.
# index表示: 行, columns表示: 列, values表示: 值, aggfunc表示: 聚合函数.
df.pivot_table(index='city', columns='gender_group', values='customer', aggfunc='sum')

In [38]:
# 需求3: 按照 城市, 销售渠道(线上, 线下)划分, 分别计算 销售金额的平均值, 成本的总和. 
# 方式1: groupby() + 聚合函数.
df.groupby(['city', 'channel']).agg({'revenue':'mean', 'unit_cost':'sum'})

# 方式2: pivot_table() 透视表
df.pivot_table(index=['city', 'channel'], values=['revenue', 'unit_cost'], aggfunc={'revenue':'mean', 'unit_cost':'sum'})

# 透视表的几种变形写法.
# 变形1: 因为没有指定哪个列 mean(平均值), 哪个列sum(总和), 所以默认: 所有列都进行 平均值和总和的计算.
df.pivot_table(index=['city', 'channel'], values=['revenue', 'unit_cost'], aggfunc=['mean', 'sum'])

# 野路子之王
df.pivot_table(index=['city', 'channel'], values=['revenue', 'unit_cost'], aggfunc=['mean', 'sum']).iloc[:, [0,3]].set_axis(['收入平均值', '成本总和'], axis=1)  # 只显示需要的列

# 变形2: 透视表, 行列转置.  columns: 分组字段的值 作为列名, values: 聚合函数的结果作为值.
df.pivot_table(index='city', columns='channel', values=['revenue', 'unit_cost'], aggfunc={'revenue':'mean', 'unit_cost':'sum'})

revenue             unit_cost          
channel          线上          线下        线上        线下
city                                               
上海       187.603426  154.623043   27980.0   82359.0
北京              NaN  226.098128       NaN   26163.0
南京              NaN  246.301860       NaN   23140.0
广州       153.470817  133.368817   59631.0   41091.0
成都              NaN  136.160798       NaN   72041.0
杭州              NaN  155.751252       NaN  173445.0
武汉       180.977961  153.258971   69225.0   93858.0
深圳              NaN  167.993511       NaN  201526.0
西安       128.581239  131.791838   10246.0   64469.0
重庆       144.672253  147.764673    8028.0   75055.0

# 3.分组过滤

In [40]:
# 需求: 按照城市分组, 查询每组销售金额平均值.
# df.groupby('city').revenue.mean()
# df.groupby('city').revenue.agg('mean')
df.groupby('city').agg({'revenue': 'mean'})

,revenue
city,
上海,163.037110
北京,226.098128
南京,246.301860
广州,145.395105
成都,136.160798
杭州,155.751252
武汉,165.342803
深圳,167.993511
西安,131.323751


In [44]:
# filter()函数, 分组过滤的.
# 需求: 按照城市分组, 查询每组销售金额平均值 大于 200的全部数据. 
df.groupby('city').get_group('上海')      # 根据分组名, 获取该分组的数据. 
df.groupby('city').get_group('深圳')      # 根据分组名, 获取该分组的数据.
df.groupby('city').get_group('广州')      # 根据分组名, 获取该分组的数据.
df.groupby('city').get_group('北京')      # 根据分组名, 获取该分组的数据.

# 大白话解释需求: 按照城市分组, 计算每组的销售金额的平均值, 并筛选出: 均值大于200的 所有分组的数据.
# filter(): 根据条件, 筛选出合法的数据. 
# 换换成SQL思路: select * from df where city in ('北京', '南京');
# 换换成SQL思路: select * from df where city in (select city from df group by city having avg(revenue) > 200);
df.groupby('city').filter(lambda s: s['revenue'].mean() > 200)
df.groupby('city')['revenue'].filter(lambda s: s.mean() > 200)

9        199.0
23       176.0
40       198.0
62       197.0
168      447.0
         ...  
22182     79.0
22192    693.0
22251     39.0
22273    288.0
22280     59.0
Name: revenue, Length: 1077, dtype: float64